# Properties of the exponential distribution (class 07)

Two properties checked by simulation against the closed-form results: the probability that one exponential variable is smaller than another, and the Erlang (gamma) distribution of a sum of exponentials, first with a loop and then vectorised with NumPy.

### Import libraries

In [24]:
import time
import numpy as np
import scipy.stats as st

In [29]:
import numpy as np
import scipy.stats as st


def p_exp_smaller(MU1, MU2, nSim):
    # probability that an exponential with mean MU1 is smaller than one with mean MU2
    exp1 = st.expon.rvs(scale=MU1, size=nSim)
    exp2 = st.expon.rvs(scale=MU2, size=nSim)
    smaller = exp1 < exp2
    return np.count_nonzero(smaller) / nSim


def sum_exp_cdf_iterative(x, N, MU, nSim):
    # CDF of the sum of N exponentials with mean MU, one simulation at a time
    hits = 0
    for i in range(nSim):
        EXP = st.expon.rvs(scale=MU, size=N)
        total = sum(EXP)
        if total <= x:
            hits = hits + 1
    return hits/nSim


def sum_exp_cdf_vectorised(x, N, MU, nSim):
    # CDF of the sum of N exponentials with mean MU, all simulations at once

    # draw an nSim x N matrix of exponential values
    matrix = np.random.exponential(MU, [nSim, N])

    # sum the rows
    row_sums = np.sum(matrix, axis=1)

    # which rows have a sum smaller than x
    smaller = row_sums < x

    return np.count_nonzero(smaller) / nSim

## Probability that one exponential random variable is smaller than another
Let $X_{1}$ and $X_{2}$ be independent random variables with means ${mu}_{1} = 1/L_{1}$ and ${mu}_{2} = 1/L_{2}$.<br>

* P[$X_{1} \leq X_{2}$] = $\frac{L_{1}}{L_{1}+L_{2}}$<br><br>

This probability can be simulated with the following vectorised algorithm, implemented in the function p_exp_smaller:<br>
* draw an array ${exp}_{1}$ with $nSim$ exponential values with mean ${mu}_{1}$;<br>
* draw an array ${exp}_{2}$ with $nSim$ exponential values with mean ${mu}_{2}$;<br>
* compute the boolean array $smaller$, True where ${exp}_{1}$ < ${exp}_{2}$;<br>
* return the number of True values in $smaller$ divided by $nSim$.<br><br>

The code below compares the probability simulated by p_exp_smaller with the theoretical value.

In [26]:
mu1 = 2
mu2 = 4
nSim = 10000

probT = (1/mu1)/(1/mu1+1/mu2)
t1 = time.perf_counter()
probS = p_exp_smaller(mu1, mu2, nSim)
t2 = time.perf_counter()
print('Simulated probability:  {:.4f}'.format(probS))
print('Theoretical probability: {:.4f}'.format(probT))
print('Simulation time: {:.4f}'.format(t2-t1))

Probabilidade simulada:  0.6684
Probabilidade teórica: 0.6667
Tempo de simulação: 0.0019


## Distribution of the sum of *N* exponential random variables
Let $X_{1}, X_{2}, \cdots , X_{N}$ be independent random variables with mean $mu = 1/L$.<br>
$X = X_{1} + X_{2} + \cdots + X_{N}$ has an Erlang distribution with parameters *N* and *L*:
* $f_{X}(x)=\frac{L^{N}x^{N-1}e^{-Lx}}{\tau (N)}$
* $F_{X}(x)=1-\sum_{j=0}^{N-1}e^{-Lx}\frac{(Lx)^{j}}{j!}$

With SciPy:<br>
* P[$X \leq x$] = st.gamma.cdf(x, a=N, scale=MU)<br><br>

## Iterative algorithm
The CDF of the sum of $N$ exponential random variables with mean $MU$ can be simulated with the following iterative algorithm (SciPy and NumPy):<br>
* start the variable $hits$ at zero;<br>
* draw an array $EXP$ with $N$ exponential random values with mean $MU$;<br>
* compute $total$, the sum of $EXP$, i.e. the sum of $N$ exponential random variables with mean $MU$;<br>
* increment $hits$ when the sum is smaller than the value $x$ for which we want the CDF (passed as an argument);<br>
* return $hits$ divided by $nSim$.<br><br>

The code below compares the probability simulated by sum_exp_cdf_iterative with the CDF of the gamma variable computed by SciPy.

In [27]:
x = 12
N = 5
MU = 2
nSim = 10000

probT = st.gamma.cdf(x, a=N, scale=MU) 
t1 = time.perf_counter()
probS = sum_exp_cdf_iterative(x, N, MU, nSim)
t2 = time.perf_counter()
print('Simulated probability:  {:.4f}'.format(probS))
print('Theoretical probability: {:.4f}'.format(probT))
print('Simulation time: {:.4f}'.format(t2-t1))

Probabilidade simulada:  0.7125
Probabilidade teórica: 0.7149
Tempo de simulação: 0.5610


## Vectorised algorithm
The same CDF can be simulated with the following vectorised NumPy algorithm:<br>

Draw a matrix $EXP$ with $nSim$ rows and $N$ columns.<br>
* Note: each row is one simulation and holds the $N$ exponential values drawn with mean MU.<br>
* Hint: np.random.exponential(MU, [nSim, N])<br><br>

Compute the array $total$ with the sum of each row of $EXP$, i.e. each element is the sum of $N$ exponential random variables with mean $MU$.<br>
* Hint: np.sum() along the rows.<br><br>

Compute the boolean array $smaller$, True for each row whose sum is smaller than the value $x$ for which we want the CDF.<br>
* Hint: (total <= x)<br><br>

Return the number of True values in $smaller$ divided by $nSim$.<br>
* Hint: np.count_nonzero counts the True values.<br><br>

The code below compares the probability simulated by sum_exp_cdf_vectorised with the CDF of the gamma variable computed by SciPy.

In [31]:
x = 12
N = 5
MU = 2
nSim = 10000

probT = st.gamma.cdf(x, a=N, scale=MU) 
t1 = time.perf_counter()
probS = sum_exp_cdf_vectorised(x, N, MU, nSim)
t2 = time.perf_counter()
print('Simulated probability:  {:.4f}'.format(probS))
print('Theoretical probability: {:.4f}'.format(probT))
print('Simulation time: {:.4f}'.format(t2-t1))

Probabilidade simulada:  0.7159
Probabilidade teórica: 0.7149
Tempo de simulação: 0.0068
